# Day 20 · Hexagons — Jobs in 45 Minutes

**Question.** A 45-minute trip is the usual yardstick for a reachable job. How many jobs can a
Chicagoan reach in 45 minutes by bus and train, and how much does that depend on where they live?

**Approach.** The city is laid out in hexagons on its own address grid: a column of centers on every
half-mile line, so every arterial runs through one, rows √3/4 mile apart. From the center of every
hexagon where people live, the CTA, Metra and Pace timetables of a Tuesday (17 November 2026) are
(and the South Shore Line's) scanned for each departure between 7:30 and 8:30 a.m., every 10 minutes, and the jobs within 45
minutes door to door are counted (LEHD LODES 2023, all jobs in the six counties). Each hexagon keeps
the median of its seven counts. Walking: a straight line ×1.3 at 1.2 m/s, as on Plate XV.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | LODES workplace file and block crosswalk (cached); community areas |
| 3. Validate | Jobs, counties, blocks |
| 4. Transform | Hexagons with residents; job points; the six-county job comb for the inset |
| 5. Analyze | Connection scans from every hexagon; classes, sides |
| 6. QA | Departure times, a known trip, totals |
| 7. Export | `plate.json` |

Run top to bottom from this folder. The six-county job file (`data/raw/lodes_wac_*.csv`) is
committed; the LODES downloads and the scan results are cached in `data/processed/`. The scans take
about 15 minutes on 8 processes.

## 1. Setup

In [ ]:
from __future__ import annotations

import gzip
import json
import logging
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from atlas import Ledger, fetch, write_plate
from atlas import transit
from atlas.grid import miles_from_origin

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day20')
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = download LODES again and redo the scans

DAY = 20
LODES_YEAR = 2023
LODES = 'https://lehd.ces.census.gov/data/lodes/LODES8/il'
LODES_URL = 'https://lehd.ces.census.gov/data/#lodes'
LODES_DOC = 'https://lehd.ces.census.gov/doc/help/onthemap/LODESTechDoc.pdf'
CA_DATASET = 'igwz-8jzy'  # Boundaries - Community Areas
GTFS_URL = 'https://www.transitchicago.com/developers/gtfs/'
PULLED = date(2026, 11, 22)
COUNTIES = {'031': 'Cook', '043': 'DuPage', '089': 'Kane', '097': 'Lake', '111': 'McHenry', '197': 'Will'}
SERVICE_DAY = date(2026, 11, 17)          # a Tuesday
STARTS = list(range(7 * 3600 + 1800, 8 * 3600 + 1801, 600))   # 7:30 ... 8:30 a.m.
HORIZON = 45 * 60                        # seconds, door to door
S = 0.5                                  # miles between hexagon centers in a row (city)
S_REGION = 1.0                           # miles, the inset's six-county comb
CLASSES = [100_000, 250_000, 500_000, 1_000_000]       # jobs: five classes on the map
REGION_CLASSES = [500, 2_000, 5_000, 20_000]            # jobs per hexagon on the inset

DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
CACHE = DAY_DIR / 'data' / 'processed'
JOBS_CSV = RAW_DIR / f'lodes_wac_{LODES_YEAR}_six_counties.csv'
CA_CSV = RAW_DIR / f'community_areas_{PULLED.isoformat()}.csv'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
GTFS = REPO / 'reference' / 'raw' / 'gtfs'
FEEDS = {'c': GTFS / 'cta_gtfs_2026-11-08.zip', 'm': GTFS / 'metra_gtfs_2026-11-05.zip', 'p': GTFS / 'pace_gtfs_2026-11-05.zip',
         'n': GTFS / 'nictd_gtfs_2026-04-10.zip'}   # South Shore Line (NICTD), feed 2026_Monon_v3, via the Mobility Database (mdb-585)
NICTD_URL = 'https://mobilitydatabase.org/feeds/gtfs/mdb-585'
BLOCKS = REPO / 'reference' / 'chicago-blocks-2020.csv'
assert (DAY_DIR / 'day20.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'
# Chicago's nine sides, by community area (as Plates X, XVII and XIX): Wikipedia's grouping (unofficial)
SIDES = {
    'Central': [8, 32, 33],
    'North Side': [5, 6, 7, 21, 22],
    'Far North Side': [1, 2, 3, 4, 9, 10, 11, 12, 13, 14, 76, 77],
    'Northwest Side': [15, 16, 17, 18, 19, 20],
    'West Side': [23, 24, 25, 26, 27, 28, 29, 30, 31],
    'South Side': [34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 60, 68, 69],
    'Southwest Side': [56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67],
    'Far Southwest Side': [70, 71, 72, 73, 74, 75],
    'Far Southeast Side': [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55],
}
SIDE_OF = {ca: s for s, cas in SIDES.items() for ca in cas}
assert sorted(SIDE_OF) == list(range(1, 78))

## 2. Pull
LODES 8 workplace area characteristics (WAC), all jobs (segment S000, job type JT00), and the LODES block crosswalk for block coordinates. Only the six counties' blocks with jobs are kept, in a small committed file.

In [ ]:
def download(url: str, path: Path) -> Path:
    if REFRESH or not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        log.info('Downloading %s', url)
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            path.write_bytes(r.content)
    return path


if REFRESH or not JOBS_CSV.exists():
    wac = download(f'{LODES}/wac/il_wac_S000_JT00_{LODES_YEAR}.csv.gz', CACHE / f'il_wac_S000_JT00_{LODES_YEAR}.csv.gz')
    xw = download(f'{LODES}/il_xwalk.csv.gz', CACHE / 'il_xwalk.csv.gz')
    w = pd.read_csv(wac, dtype={'w_geocode': str}, usecols=['w_geocode', 'C000'])
    x = pd.read_csv(xw, dtype={'tabblk2020': str, 'cty': str}, usecols=['tabblk2020', 'cty', 'blklatdd', 'blklondd'])
    x['county'] = x['cty'].str[-3:]
    jobs = w.merge(x, left_on='w_geocode', right_on='tabblk2020', how='left', validate='one_to_one')
    assert jobs['county'].notna().all(), 'A workplace block is missing from the crosswalk'
    jobs = jobs[jobs['county'].isin(COUNTIES)][['w_geocode', 'county', 'C000', 'blklatdd', 'blklondd']]
    jobs.columns = ['block', 'county', 'jobs', 'lat', 'lon']
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    jobs.to_csv(JOBS_CSV, index=False)
jobs = pd.read_csv(JOBS_CSV, dtype={'block': str, 'county': str})

if REFRESH or not CA_CSV.exists():
    fetch(CA_DATASET, select='area_numbe,community').to_csv(CA_CSV, index=False)
ca_names = pd.read_csv(CA_CSV)
CA_NAME = {int(r.area_numbe): r.community.title().replace("'S", "'s").replace('Ohare', "O'Hare") for r in ca_names.itertuples()}
len(jobs), jobs['jobs'].sum()

## 3. Validate

In [ ]:
assert jobs['block'].is_unique
assert (jobs['jobs'] > 0).all()
assert set(jobs['county']) == set(COUNTIES)
assert len(CA_NAME) == 77
by_county = jobs.groupby('county')['jobs'].sum().rename(index=COUNTIES).sort_values(ascending=False)
blocks = pd.read_csv(BLOCKS)
assert blocks['pop'].sum() == 2_746_388
print(f'{jobs["jobs"].sum():,} jobs in {len(jobs):,} blocks')
by_county

## 4. Transform
Hexagons on the address grid, pointy side up: centers every half mile along a row (x = 0, ½, 1 ... miles
from State Street), rows √3/4 mile apart from Madison, every other row shifted a quarter mile. Each
census block goes to its nearest center; a hexagon is kept if anyone lives in it.

In [ ]:
def hex_of(x, y, s):
    """Nearest pointy-top lattice center for points in miles: (row, col) arrays."""
    rh = s * np.sqrt(3) / 2
    r0 = np.floor(y / rh).astype(int)
    best = None
    for dr in (0, 1):
        r = r0 + dr
        off = (r % 2) * s / 2
        c = np.round((x - off) / s).astype(int)
        d = np.hypot(x - (c * s + off), y - r * rh)
        if best is None:
            best = [d, r, c]
        else:
            m = d < best[0]
            best = [np.where(m, d, best[0]), np.where(m, r, best[1]), np.where(m, c, best[2])]
    return best[1], best[2]


def hex_center(r, c, s):
    return c * s + (r % 2) * s / 2, r * s * np.sqrt(3) / 2


blocks['x'], blocks['y'] = miles_from_origin(blocks['lat'].to_numpy(), blocks['lon'].to_numpy())
blocks['r'], blocks['c'] = hex_of(blocks['x'].to_numpy(), blocks['y'].to_numpy(), S)
blocks['side'] = blocks['commarea'].map(SIDE_OF)
cells = blocks.groupby(['r', 'c']).agg(pop=('pop', 'sum')).reset_index()
# each hexagon's community area: the one most of its residents live in (the portal's blocks; 33 edge blocks have none)
ca_major = (blocks[blocks['commarea'] > 0].groupby(['r', 'c', 'commarea'])['pop'].sum().reset_index()
            .sort_values('pop').drop_duplicates(['r', 'c'], keep='last').set_index(['r', 'c'])['commarea'])
cells['ca'] = [int(ca_major.get((r, c), 0)) for r, c in zip(cells['r'], cells['c'])]
cells['x'], cells['y'] = hex_center(cells['r'].to_numpy(), cells['c'].to_numpy(), S)
jobs['x'], jobs['y'] = miles_from_origin(jobs['lat'].to_numpy(), jobs['lon'].to_numpy())
print(f'{len(cells):,} hexagons with residents; area {S * S * np.sqrt(3) / 2:.4f} sq mi each')
cells.describe().round(2)

In [ ]:
# The inset: every job in the six counties on a coarser comb (centers a mile apart)
jobs['R'], jobs['C'] = hex_of(jobs['x'].to_numpy(), jobs['y'].to_numpy(), S_REGION)
region = jobs.groupby(['R', 'C']).agg(jobs=('jobs', 'sum')).reset_index()
cty_major = jobs.groupby(['R', 'C', 'county'])['jobs'].sum().reset_index().sort_values('jobs').drop_duplicates(['R', 'C'], keep='last')
region = region.merge(cty_major[['R', 'C', 'county']], on=['R', 'C'])
print(f'{len(region):,} region hexagons; jobs per hexagon quantiles:')
region['jobs'].quantile([0.25, 0.5, 0.75, 0.9, 0.99]).round()

## 5. Analyze
One connection scan per hexagon and departure: the earliest arrival at every stop, then the walk to every
job block within ¾ mile of a stop (or straight from the hexagon). A job counts if it is reached within
45 minutes. The hexagon keeps the median over the seven departures.

In [ ]:
cache = CACHE / f'reach_{LODES_YEAR}_{FEEDS["c"].stem.split("_")[-1]}_{len(FEEDS)}feeds.npz'
if REFRESH or not cache.exists():
    net = transit.connections(FEEDS, SERVICE_DAY, STARTS[0], STARTS[-1] + HORIZON)
    log.info('%d connections, %d stops', len(net['conn']), len(net['x']))
    counts = transit.reach(net, cells[['x', 'y']].to_numpy(), jobs[['x', 'y']].to_numpy(), jobs['jobs'].to_numpy(),
                           STARTS, HORIZON)
    np.savez(cache, counts=counts, r=cells['r'].to_numpy(), c=cells['c'].to_numpy())
z = np.load(cache)
assert (z['r'] == cells['r'].to_numpy()).all() and (z['c'] == cells['c'].to_numpy()).all()
counts = z['counts']
cells['jobs'] = np.median(counts, axis=1)
cells['cls'] = np.searchsorted(CLASSES, cells['jobs'], side='right')
cells['side'] = cells['ca'].map(SIDE_OF)
cells[['pop', 'jobs']].describe().round()

In [ ]:
def wmedian(v, w):
    """The value half of the weight lies at or below (residents, not hexagons)."""
    o = np.argsort(v)
    cw = np.cumsum(np.asarray(w, float)[o])
    return float(np.asarray(v, float)[o][np.searchsorted(cw, cw[-1] / 2)])


TOTAL = int(jobs['jobs'].sum())
pop = cells['pop'].sum()
classes = []
for k in range(len(CLASSES) + 1):
    m = cells['cls'] == k
    classes.append({'k': k, 'lo': ([0] + CLASSES)[k], 'hi': (CLASSES + [None])[k], 'hexes': int(m.sum()),
                    'pop': int(cells.loc[m, 'pop'].sum()), 'share': float(cells.loc[m, 'pop'].sum() / pop)})
sides = []
for s in SIDES:
    d = cells[cells['side'] == s]
    sides.append({'side': s, 'pop': int(d['pop'].sum()), 'median': wmedian(d['jobs'], d['pop']),
                  'byClass': [int(d.loc[d['cls'] == k, 'pop'].sum()) for k in range(len(CLASSES) + 1)]})
sides.sort(key=lambda r: -r['median'])
cas = (cells[cells['ca'] > 0].groupby('ca').apply(lambda d: pd.Series({'pop': d['pop'].sum(), 'median': wmedian(d['jobs'], d['pop'])}))
       .sort_values('median'))
best = cells.loc[cells['jobs'].idxmax()]
summary = {
    'pop': int(pop), 'hexes': len(cells), 'jobsTotal': TOTAL, 'cookJobs': int(jobs.loc[jobs['county'] == '031', 'jobs'].sum()),
    'median': wmedian(cells['jobs'], cells['pop']),
    'under100k': float(cells.loc[cells['jobs'] < 100_000, 'pop'].sum() / pop),
    'over1M': float(cells.loc[cells['jobs'] >= 1_000_000, 'pop'].sum() / pop),
    'max': float(best['jobs']), 'maxShare': float(best['jobs'] / TOTAL), 'maxCa': CA_NAME[int(best['ca'])],
    'lowCas': [{'name': CA_NAME[int(i)], 'median': float(r['median'])} for i, r in cas.head(5).iterrows()],
    'highCas': [{'name': CA_NAME[int(i)], 'median': float(r['median'])} for i, r in cas.tail(5)[::-1].iterrows()],
}
print(json.dumps(summary, indent=1))
pd.DataFrame(sides)[['side', 'pop', 'median']]

## 6. QA

In [ ]:
# Departure time: how much the count moves within the hour (median hexagon's range, residents' median of the spread)
spread = (counts.max(axis=1) - counts.min(axis=1)) / np.maximum(1, np.median(counts, axis=1))
qa = {'spreadMedian': wmedian(spread, cells['pop'])}
# 8:00 alone against the median of seven
at8 = counts[:, STARTS.index(8 * 3600)]
qa['median8am'] = wmedian(at8, cells['pop'])
# the hexagon at State & Madison reaches the Loop's jobs on foot alone
loop = cells.loc[(cells['r'] == 0) & (cells['c'] == 0)]
qa['loopHex'] = float(loop['jobs'].iloc[0]) if len(loop) else None
# jobs per county: LODES counts jobs where people work, so Cook holds the city's and the suburbs'
qa['byCounty'] = {k: int(v) for k, v in by_county.items()}
assert summary['median'] > 0 and summary['max'] <= TOTAL
assert abs(sum(c['pop'] for c in classes) - pop) == 0
qa

## 7. Export

In [ ]:
fe = lambda v: f'{int(round(v, -4)):,}' if v >= 100_000 else f'{int(round(v, -3)):,}'
fse = next(s for s in sides if s['side'] == 'Far Southeast Side')
finding = (f'Half of Chicagoans can reach about {fe(summary["median"])} jobs within 45 minutes by bus and train. '
           f'On the Far Southeast Side, half reach about {fe(fse["median"])} or fewer.')
ledger = Ledger(
    source='US Census Bureau, LEHD Origin-Destination Employment Statistics (LODES 8), workplace area characteristics',
    dataset_id=f'il_wac_S000_JT00_{LODES_YEAR}', url=LODES_URL, pulled=PULLED,
    units='jobs (all jobs, primary and secondary)',
    method=['Hexagons on the address grid, centers every 1/2 mile, rows sqrt(3)/4 mile apart',
            'Connection scan over CTA, Metra, Pace and South Shore Line GTFS, Tue 17 Nov 2026, departures 7:30-8:30 every 10 min',
            'Jobs within 45 minutes door to door; median of the seven departures'],
    caveats=['Schedules as published: no delays, no fares, no crowding',
             'Walking as straight lines x1.3 at 1.2 m/s; jobs outside the six Illinois counties not counted'],
)
payload = {
    'S': S, 'S_region': S_REGION, 'classes': CLASSES, 'regionClasses': REGION_CLASSES,
    'lodesYear': LODES_YEAR, 'counties': COUNTIES,
    'gtfs': {'cta': FEEDS['c'].stem.split('_')[-1], 'metra': FEEDS['m'].stem.split('_')[-1], 'pace': FEEDS['p'].stem.split('_')[-1],
             'nictd': '2026_Monon_v3', 'nictdUrl': NICTD_URL,
             'day': SERVICE_DAY.isoformat(), 'url': GTFS_URL},
    'starts': ['7:30', '8:30'], 'horizon': HORIZON // 60,
    'sources': {'lodes': LODES_URL, 'lodesDoc': LODES_DOC, 'ca': f'https://data.cityofchicago.org/d/{CA_DATASET}'},
    'summary': summary, 'qa': qa, 'classTable': classes, 'sides': sides,
    'caNames': {str(k): v for k, v in CA_NAME.items()},
    'hexCols': ['r', 'c', 'pop', 'jobs', 'ca'],
    'hexes': [[int(r.r), int(r.c), int(r.pop), int(round(r.jobs)), int(r.ca)] for r in cells.itertuples()],
    'regionCols': ['r', 'c', 'jobs', 'county'],
    'region': [[int(r.R), int(r.C), int(r.jobs), r.county] for r in region.itertuples()],
}
write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=finding, payload=payload)
log.info('Wrote %s (%.0f KB)', OUT_PATH, OUT_PATH.stat().st_size / 1024)
finding